# 05 离散数据积分

函数型积分允许我们主动选择节点；离散数据积分则只能使用已有采样点。此时问题从“如何调用函数”变成“如何在数据之间建立合理的局部模型”。

本节讨论梯形积分、局部二次 Simpson 积分、平均抛物线积分、自然三次样条积分，以及噪声对积分结果的影响。


In [ ]:
import math
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

plt.rcParams["font.sans-serif"] = [
    "Arial Unicode MS",
    "PingFang SC",
    "Heiti SC",
    "SimHei",
    "Noto Sans CJK SC",
]
plt.rcParams["axes.unicode_minus"] = False

ROOT = pathlib.Path.cwd().resolve()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "pyproject.toml").exists() and (candidate / "src" / "py_sc").exists():
        ROOT = candidate
        break
SRC = ROOT / "src"
CHAPTER = ROOT / "chapters" / "ch04_numerical_integration"
SCRIPTS = CHAPTER / "scripts"
for path in [SRC, SCRIPTS]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from py_sc import (
    average_parabolic_integral,
    discrete_simpson,
    discrete_trapezoid,
    natural_cubic_spline_integral,
)

try:
    from scipy import integrate as scipy_integrate
    SCIPY_AVAILABLE = True
except Exception:
    SCIPY_AVAILABLE = False


## 梯形积分

对离散数据

$$
(x_0,y_0),\ldots,(x_n,y_n),
$$

最稳健的做法是相邻点之间用直线连接：

$$
\int_{x_0}^{x_n} f(x)\,dx
\approx
\sum_{i=0}^{n-1}\frac{y_i+y_{i+1}}{2}(x_{i+1}-x_i).
$$

这个公式允许非等距节点，但只有局部线性精度。


In [ ]:
x_data = np.array([0.0, 0.15, 0.31, 0.55, 0.72, 1.0])
y_data = np.exp(-x_data**2)
trap_value = discrete_trapezoid(x_data, y_data)
print("非等距离散梯形积分:", trap_value)


## 局部二次插值和 Simpson

若每三个相邻点构造一个二次插值多项式，再对该多项式积分，就得到局部二次积分。等距且面板数为偶数时，它就是复合 Simpson 公式。

对非等距节点，仍可直接解三点二次插值系数：

$$
q(x)=c_0+c_1x+c_2x^2,
$$

然后精确计算

$$
\int_\alpha^\beta q(x)\,dx
=c_0(\beta-\alpha)+\frac{c_1}{2}(\beta^2-\alpha^2)+\frac{c_2}{3}(\beta^3-\alpha^3).
$$


In [ ]:
def teaching_quadratic_panel_integral(x3, y3):
    coefficients = np.linalg.solve(np.vander(x3, 3, increasing=True), y3)
    left, right = x3[0], x3[-1]
    powers = np.array([right - left, (right**2 - left**2) / 2.0, (right**3 - left**3) / 3.0])
    return float(np.dot(coefficients, powers))

x_quad = np.array([-1.0, -0.2, 0.8])
y_quad = x_quad**2 + 2 * x_quad + 1
print("二次函数面板积分:", teaching_quadratic_panel_integral(x_quad, y_quad))
print("精确值:", (0.8**3 - (-1.0)**3) / 3 + (0.8**2 - (-1.0)**2) + (0.8 - (-1.0)))


## 样条积分

三次样条先在全部节点上构造分段三次函数

$$
s_i(x)=a_i+b_i(x-x_i)+c_i(x-x_i)^2+d_i(x-x_i)^3,
$$

再逐段精确积分：

$$
\int_{x_i}^{x_{i+1}}s_i(x)\,dx
=a_i h_i+\frac{b_i}{2}h_i^2+\frac{c_i}{3}h_i^3+\frac{d_i}{4}h_i^4.
$$

样条积分利用了更多全局光滑性信息，但对含噪声数据也可能把噪声变成曲线摆动。


In [ ]:
x = np.linspace(0.0, math.pi, 17)
y = np.sin(x)
exact = 2.0
values = {
    "梯形": discrete_trapezoid(x, y),
    "局部二次 Simpson": discrete_simpson(x, y),
    "平均抛物线": average_parabolic_integral(x, y),
    "自然三次样条": natural_cubic_spline_integral(x, y),
}
if SCIPY_AVAILABLE:
    values["SciPy simpson"] = scipy_integrate.simpson(y, x=x)

print("方法              结果           误差")
for name, value in values.items():
    print(f"{name:14s} {value: .12f} {abs(value-exact): .3e}")


## 噪声数据的影响

高阶局部模型并不总是更好。若数据含噪声，积分误差同时来自采样间距和噪声。积分本身有一定平均作用，但过度追随噪声的插值曲线仍可能引入偏差。


In [ ]:
rng = np.random.default_rng(2026)
x_noisy = np.linspace(0.0, math.pi, 31)
y_true = np.sin(x_noisy)
y_noisy = y_true + 0.04 * rng.normal(size=x_noisy.size)

methods = {
    "梯形": discrete_trapezoid(x_noisy, y_noisy),
    "Simpson": discrete_simpson(x_noisy, y_noisy),
    "平均抛物线": average_parabolic_integral(x_noisy, y_noisy),
    "自然三次样条": natural_cubic_spline_integral(x_noisy, y_noisy),
}

plt.figure(figsize=(8, 4.4))
xx = np.linspace(0.0, math.pi, 500)
plt.plot(xx, np.sin(xx), label="真实函数")
plt.plot(x_noisy, y_noisy, "o", label="含噪声采样")
plt.xlabel("x")
plt.ylabel("y")
plt.title("离散数据积分面对的是采样和噪声")
plt.legend()
plt.show()

plt.figure(figsize=(7, 4.0))
plt.bar(methods.keys(), [abs(v - 2.0) for v in methods.values()])
plt.ylabel("绝对误差")
plt.title("含噪声数据上的积分误差")
plt.xticks(rotation=15)
plt.show()

for name, value in methods.items():
    print(f"{name:10s} value={value:.8f}, error={abs(value-2.0):.3e}")


## 适用条件、失败情形和小结

* 梯形积分最稳健，适合非等距节点和噪声数据的基础估计。
* Simpson 或局部二次积分对光滑无噪声数据更准确，但通常要求合适的节点数量或三点面板。
* 平均抛物线积分可以减少局部二次面板选择造成的偏向，但不是严格误差最优方法。
* 样条积分适合光滑曲线数据；若数据噪声较大，应考虑先做平滑或使用统计模型。
* 离散数据积分不能简单套用可调用函数积分的自适应策略，因为缺失的函数值无法凭空获得。
